# Маркет ВКонтакте и партнёр: итоги и роль Маркета

Период — 30 дней сентября 2026 года. Сравниваются агрегаты Маркета VK и условного внешнего партнёра; пути заказа могут различаться. Источник синтетический; показатели каналов и их пользователей могут пересекаться. Для повторного запуска откройте ноутбук из папки `notebooks`: CSV ожидаются в соседней папке `Data`.

**Важно:** прямые переходы у Маркета и оплаты у партнёра в исходных данных заполнены нулём как неприменимые показатели; эти две ячейки переводятся в пропуски, остальные нули сохраняются.

## Как читать маршруты: методологически корректная трактовка

- **Карточка Маркета VK:** открытие карточки товара Маркета во VK, затем действие пользователя. Нажатие на кнопку может вести к чекауту внутри VK, в чат с продавцом или на другой адрес — фактический справочник назначения кнопок в выгрузке отсутствует.
- **Карточка партнёра во VK:** открытие карточки товара партнёра во VK и последующее нажатие на кнопку. Нажатие может вести к разным следующим шагам, включая оформление на площадке партнёра. Сам факт нажатия не означает автоматически оформленный или оплаченный заказ.
- **Прямой маршрут партнёра:** переход к партнёру без открытия карточки товара во VK. В данных он измеряется отдельным событием и относится к иной когорте.

Партнёрская карточка и прямой переход партнёра нельзя складывать в уникальных покупателей или конверсию. Маркет и партнёра следует сравнивать как связанные, но не одинаковые продуктовые маршруты. Для выводов нужны когортные данные по назначению кнопки, чекауту и оплате.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

data_path = Path.cwd().parent / 'Data'
mv = 'Маркет Вконтакте'
p1 = 'Партнёр 1'
pd.set_option('display.float_format', '{:,.3f}'.format)

market = pd.read_csv(data_path / 'Market_Partner.csv').set_index('Метрика')
total = pd.read_csv(data_path / 'Market_Total.csv').set_index('Период')
market.loc[['Прямые переходы к Партнёру 1 (без открытия карточки)'], mv] = market.loc[['Прямые переходы к Партнёру 1 (без открытия карточки)'], mv].replace(0, np.nan)
market.loc[['Оплаченные заказы'], p1] = market.loc[['Оплаченные заказы'], p1].replace(0, np.nan)
market

,Маркет Вконтакте,Партнёр 1
Метрика,,
Показы товаров,"4,777,750,023.000","5,729,628,790.000"
Открытия карточки товара,"193,512,461.000","86,882,822.000"
Нажатия на кнопку в карточке,"5,969,063.000","36,626,075.000"
Прямые переходы к Партнёру 1 (без открытия карточки),NaN,"106,914,679.000"
Оформленные заказы,"413,740.000","4,927,525.000"
Оплаченные заказы,"4,903.000",NaN
"GMV, ₽","1,564,546,740.000","5,119,514,685.000"
"Средний чек, ₽","3,781.000","1,039.000"
Уникальные покупатели за 30 дней,"296,148.000","2,901,334.000"


In [2]:
# Интерактивные визуализации
import plotly.graph_objects as go

def show_plotly(fig, title, x_title, y_title, height=500):
    fig.update_layout(
        title=dict(text=title), template='plotly_white', height=height,
        xaxis_title=x_title, yaxis_title=y_title,
        legend_title_text='Канал', margin=dict(l=70, r=30, t=80, b=90),
        font=dict(size=13), hoverlabel=dict(namelength=-1),
    )
    fig.show(config={'displaylogo': False, 'responsive': True})


## Проверка производных показателей

Сопоставляем приведённые в таблице показатели с пересчётом из абсолютных значений. В исходной таблице показатели округлены. Пропуски означают отсутствие сопоставимого значения, а не нулевой результат.

In [3]:
derived = pd.DataFrame(index=['Средний чек, ₽', 'Заказов на покупателя', 'Конверсия: показ → открытие карточки, %', 'Конверсия: открытие карточки → заказ, %', 'Конверсия: показ → заказ, %'], columns=[mv, p1], dtype=float)
derived.loc['Средний чек, ₽'] = market.loc['GMV, ₽'] / market.loc['Оформленные заказы']
derived.loc['Заказов на покупателя'] = market.loc['Оформленные заказы'] / market.loc['Уникальные покупатели за 30 дней']
derived.loc['Конверсия: показ → открытие карточки, %'] = market.loc['Открытия карточки товара'] / market.loc['Показы товаров'] * 100
derived.loc['Конверсия: открытие карточки → заказ, %'] = market.loc['Оформленные заказы'] / market.loc['Открытия карточки товара'] * 100
derived.loc['Конверсия: показ → заказ, %'] = market.loc['Оформленные заказы'] / market.loc['Показы товаров'] * 100
pd.concat({'В данных': market.loc[derived.index], 'Пересчёт': derived}, axis=1)

В данных            \
                                        Маркет Вконтакте Партнёр 1   
Средний чек, ₽                                 3,781.000 1,039.000   
Заказов на покупателя                              1.400     1.700   
Конверсия: показ → открытие карточки, %            4.050     1.520   
Конверсия: открытие карточки → заказ, %            0.210     5.670   
Конверсия: показ → заказ, %                        0.009     0.086   

                                                Пересчёт            
                                        Маркет Вконтакте Партнёр 1  
Средний чек, ₽                                 3,781.473 1,038.963  
Заказов на покупателя                              1.397     1.698  
Конверсия: показ → открытие карточки, %            4.050     1.516  
Конверсия: открытие карточки → заказ, %            0.214     5.671  
Конверсия: показ → заказ, %                        0.009     0.086

## Масштаб и эффективность

Доля Маркета — его показатель, делённый на сумму показателей двух каналов, а не доля уникального рынка. Воронки не являются строгими когортными: один человек может совершить несколько действий. У партнёра заказы могут приходить и из прямых переходов к странице товара вне VK, в обход карточки VK, поэтому отношение заказов к открытиям не является чистой конверсией одной и той же когорты.

In [4]:
volume_metrics = ['Показы товаров', 'Открытия карточки товара', 'Нажатия на кнопку в карточке', 'Оформленные заказы', 'GMV, ₽', 'Уникальные покупатели за 30 дней']
volumes = market.loc[volume_metrics].copy()
volumes['Доля Маркета в сумме каналов, %'] = volumes[mv] / (volumes[mv] + volumes[p1]) * 100
volumes

,Маркет Вконтакте,Партнёр 1,"Доля Маркета в сумме каналов, %"
Метрика,,,
Показы товаров,"4,777,750,023.000","5,729,628,790.000",45.470
Открытия карточки товара,"193,512,461.000","86,882,822.000",69.014
Нажатия на кнопку в карточке,"5,969,063.000","36,626,075.000",14.013
Оформленные заказы,"413,740.000","4,927,525.000",7.746
"GMV, ₽","1,564,546,740.000","5,119,514,685.000",23.407
Уникальные покупатели за 30 дней,"296,148.000","2,901,334.000",9.262


### Интерактивно: доля Маркета в сумме двух каналов
Показаны разные показатели в одинаковой относительной шкале. Покупатели каналов могут пересекаться; сумма не является числом уникальных людей или размером рынка.


In [5]:
fig = go.Figure(go.Bar(
    x=volumes['Доля Маркета в сумме каналов, %'].iloc[::-1],
    y=volumes.index[::-1], orientation='h', marker_color='#3978C6',
    customdata=volumes[[mv, p1]].iloc[::-1].to_numpy(),
    hovertemplate='%{y}<br>Доля Маркета: %{x:.2f}%<br>Маркет: %{customdata[0]:,.0f}<br>Партнёр: %{customdata[1]:,.0f}<extra></extra>'
))
fig.update_xaxes(range=[0, 100], ticksuffix='%')
show_plotly(fig, 'Масштаб Маркета относительно суммы каналов', 'Доля Маркета в сумме двух каналов, %', '', 500)


In [6]:
efficiency = pd.DataFrame(index=['Открытия / показы, %', 'Нажатия / открытия, %', 'Заказы / открытия, %', 'Заказы / показы, %', 'GMV на 1 000 показов, ₽', 'GMV на открытие, ₽', 'GMV на покупателя, ₽', 'Заказов на покупателя', 'Повторные покупатели, %'], columns=[mv, p1], dtype=float)
efficiency.loc['Открытия / показы, %'] = market.loc['Открытия карточки товара'] / market.loc['Показы товаров'] * 100
efficiency.loc['Нажатия / открытия, %'] = market.loc['Нажатия на кнопку в карточке'] / market.loc['Открытия карточки товара'] * 100
efficiency.loc['Заказы / открытия, %'] = market.loc['Оформленные заказы'] / market.loc['Открытия карточки товара'] * 100
efficiency.loc['Заказы / показы, %'] = market.loc['Оформленные заказы'] / market.loc['Показы товаров'] * 100
efficiency.loc['GMV на 1 000 показов, ₽'] = market.loc['GMV, ₽'] / market.loc['Показы товаров'] * 1000
efficiency.loc['GMV на открытие, ₽'] = market.loc['GMV, ₽'] / market.loc['Открытия карточки товара']
efficiency.loc['GMV на покупателя, ₽'] = market.loc['GMV, ₽'] / market.loc['Уникальные покупатели за 30 дней']
efficiency.loc['Заказов на покупателя'] = market.loc['Оформленные заказы'] / market.loc['Уникальные покупатели за 30 дней']
efficiency.loc['Повторные покупатели, %'] = market.loc['Доля повторных покупателей (2+ заказа), %']
efficiency

,Маркет Вконтакте,Партнёр 1
"Открытия / показы, %",4.050,1.516
"Нажатия / открытия, %",3.085,42.156
"Заказы / открытия, %",0.214,5.671
"Заказы / показы, %",0.009,0.086
"GMV на 1 000 показов, ₽",327.465,893.516
"GMV на открытие, ₽",8.085,58.924
"GMV на покупателя, ₽","5,282.989","1,764.538"
Заказов на покупателя,1.397,1.698
"Повторные покупатели, %",19.000,26.600


### Интерактивно: отношения событий
Это отношения агрегированных событий, а не когортные конверсии. Заказы партнёра включают прямые переходы, в том числе без открытия карточки.

Различие 3,08% и 42,16% не означает сопоставимую конверсию: кнопки ведут к разным следующим шагам. Разбивки по назначению и связанной когорты нет.


In [7]:
rate_names = ['Открытия / показы, %', 'Нажатия / открытия, %', 'Заказы / показы, %']
fig = go.Figure()
for channel, color in [(mv, '#3978C6'), (p1, '#F2994A')]:
    fig.add_bar(name=channel, x=rate_names, y=efficiency.loc[rate_names, channel], marker_color=color,
                hovertemplate='%{x}<br>%{fullData.name}: %{y:.2f}%<extra></extra>')
fig.update_layout(barmode='group')
fig.update_yaxes(rangemode='tozero', ticksuffix='%')
show_plotly(fig, 'Сопоставление отношений событий', '', 'Отношение событий, %', 460)


In [8]:
direct = market.loc['Прямые переходы к Партнёру 1 (без открытия карточки)', p1]
routes = pd.Series({
    'Партнёр: нажатия на кнопку карточки VK': market.loc['Нажатия на кнопку в карточке', p1],
    'Партнёр: прямые переходы без карточки VK': direct,
    'Маркет: нажатия на кнопку карточки VK': market.loc['Нажатия на кнопку в карточке', mv],
}, name='Событий за 30 дней')
routes


Партнёр: нажатия на кнопку карточки VK      36,626,075.000
Партнёр: прямые переходы без карточки VK   106,914,679.000
Маркет: нажатия на кнопку карточки VK        5,969,063.000
Name: Событий за 30 дней, dtype: float64

### Выводы

- На Маркет приходится около 69% открытий карточек двух каналов, но 7,7% оформленных заказов и 23,4% GMV. Средний чек Маркета около 3 781 ₽ против 1 039 ₽ у партнёра.
- Нажатия относительно открытий — 3,1% у Маркета и 42,2% у партнёра. Кнопки ведут к разным следующим шагам, а прямой маршрут партнёра обходит карточку VK. Эти отношения нельзя считать сопоставимыми конверсиями или доказательством проблем чекаута.
- 4 903 оплаты составляют 1,19% оформленных заказов Маркета. Это не доказывает, что остальные заказы не были оплачены вне ВК. Сначала проверить учёт оплат и маршруты; проверка условий чекаута — гипотеза, а не диагноз.
- На одного покупателя Маркет генерирует больше GMV, однако покупателей и заказов существенно меньше. При партнёрской модели часть покупок происходит вне ВК; сравнение не измеряет все внешние продажи бизнеса.

## Охват отдельного Маркета

MAU отдельной витрины, увидевшие товары и открывшие карточку относятся к разным, возможно пересекающимся аудиториям. Соотношение ниже — масштаб относительно охвата, не доказанная доля одной когорты.

In [9]:
total


,Значение
Период,
DAU Маркета Вконтакте (среднее за день),"127,406.000"
MAU Маркета Вконтакте (уникальные за 30 дней),"2,997,341.000"
"Пользователи, видевшие товары за 30 дней","66,617,967.000"
"Пользователи, открывавшие карточку товара за 30 дней","27,333,960.000"
Открытия карточки товара из постов и клипов,"86,321,893.000"
в т.ч. из реферальных постов и клипов,"82,324,081.000"
Запросы в товарном поиске,"28,585,243.000"
"Доля поисковых запросов с открытием карточки, %",18.400


In [10]:
reach = pd.Series({'MAU Маркета / видевшие товары, %': total.loc['MAU Маркета Вконтакте (уникальные за 30 дней)', 'Значение'] / total.loc['Пользователи, видевшие товары за 30 дней', 'Значение'] * 100, 'Открывавшие / видевшие товары, %': total.loc['Пользователи, открывавшие карточку товара за 30 дней', 'Значение'] / total.loc['Пользователи, видевшие товары за 30 дней', 'Значение'] * 100, 'Средний DAU / MAU, %': total.loc['DAU Маркета Вконтакте (среднее за день)', 'Значение'] / total.loc['MAU Маркета Вконтакте (уникальные за 30 дней)', 'Значение'] * 100})
reach

MAU Маркета / видевшие товары, %    4.499
Открывавшие / видевшие товары, %   41.031
Средний DAU / MAU, %                4.251
dtype: float64

MAU отдельного Маркета — около 3 млн против 66,6 млн пользователей, видевших товары. В сочетании с малой долей открытий с главной Маркета (см. ноутбук 03) это поддерживает гипотезу об инфраструктурной роли каталога, поиска, карточек и чекаута внутри ВК, а не о ставке только на отдельную витрину. Доказать, что витрина не нужна, эти агрегаты не позволяют.

## Дополнительная проверка маршрутов

Перед ростом чекаута сначала нужно различать три вопроса: куда ведёт конкретная кнопка, дошёл ли пользователь до оформления и был ли заказ оплачен. Таблица ниже показывает объёмы разных событий, но не последовательность одной когорты.

In [11]:
route_events = pd.Series({
    'Маркет: нажатия на кнопку карточки': market.loc['Нажатия на кнопку в карточке', mv],
    'Партнёр: нажатия на кнопку карточки': market.loc['Нажатия на кнопку в карточке', p1],
    'Партнёр: прямые переходы без карточки': market.loc['Прямые переходы к Партнёру 1 (без открытия карточки)', p1],
    'Маркет: оформленные заказы': market.loc['Оформленные заказы', mv],
    'Партнёр: оформленные заказы': market.loc['Оформленные заказы', p1],
}).sort_values()

import plotly.graph_objects as go
fig = go.Figure(go.Bar(
    x=route_events.values,
    y=route_events.index,
    orientation='h',
    marker=dict(color=['#B8C2CC', '#3978C6', '#3978C6', '#70A7C6', '#70A7C6']),
    text=[f'{v/1e6:.2f} млн' if v >= 1e6 else f'{v/1e3:.1f} тыс.' for v in route_events.values],
    textposition='outside',
    hovertemplate='%{y}<br>%{x:,.0f}<extra></extra>'
))
fig.update_layout(template='plotly_white', height=430, margin=dict(l=180, r=40, t=70, b=50),
                  title='Разные маршруты нельзя объединять в одну воронку',
                  xaxis_title='Событий или заказов за 30 дней')
fig.show(config={'displaylogo': False, 'responsive': True})